# NB7 — LOO Error Analysis (artifact-safe)

Mục tiêu: giải thích **vì sao LOO Top-1 thấp hơn Hit@2** và phân biệt:

- **near-miss / ambiguity**: ground-truth đứng rank 2 nhưng \(\Delta\) rất sát Top-1;
- **clear ranking error**: ground-truth bị một item khác vượt xa;
- **length effect**: localization thay đổi theo outfit length;
- **data-ground-truth ambiguity**: synthetic swapped item không nhất thiết là unique worst item theo scorer.

Notebook **không train lại model** và **không chạy lại LOO**. Nó phân tích `loo_predictions_valid.jsonl` và `loo_predictions_test.jsonl` do NB6B tạo.

> Lưu ý Colab: `/content/...` là ephemeral. Nếu runtime NB6B đã reset thì các file evaluation cũ sẽ mất trừ khi đã copy sang Google Drive. NB7 v2 tự tìm ở local + Google Drive và chỉ hard-fail sau khi đã thử các vị trí này.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from collections import Counter, defaultdict

EXPECTED_BRANCH = "exp/min2-scorer-loo3"
REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"

def is_project_repo(path: Path) -> bool:
    path = path.expanduser().resolve()
    return (
        (path / ".git").exists()
        and (path / "src/diagnosis/evaluate_loo.py").is_file()
        and (path / "configs/data_paths.min2_experiment.json").is_file()
    )

candidates = []
if os.environ.get("FASHION_PROJECT_ROOT"):
    candidates.append(Path(os.environ["FASHION_PROJECT_ROOT"]))

cwd = Path.cwd().resolve()
candidates.extend([cwd, *cwd.parents])
if Path("/content").exists():
    candidates.append(Path("/content/opisoverated"))
candidates.append(Path.home() / "opisoverated")

ROOT = next((p.expanduser().resolve() for p in candidates if is_project_repo(p)), None)

if ROOT is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.home()
    ROOT = clone_parent / "opisoverated"
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(
            f"{ROOT} exists but is not a usable checkout. "
            "Remove/rename it or set FASHION_PROJECT_ROOT."
        )
    subprocess.run(
        ["git", "clone", "--branch", EXPECTED_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )

branch = subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()
if branch != EXPECTED_BRANCH:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", EXPECTED_BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", EXPECTED_BRANCH], check=True)

subprocess.run(
    ["git", "-C", str(ROOT), "pull", "--ff-only", "origin", EXPECTED_BRANCH],
    check=True,
)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo root :", ROOT)
print("branch    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip())
print("commit    :", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True
).strip())


## 1. Locate NB6B evaluation artifacts

NB6B tạo tối thiểu:

```text
evaluation_min2_exp_v1/
├── evaluation_summary.json          # optional cho NB7
├── loo_predictions_valid.jsonl      # required
└── loo_predictions_test.jsonl       # required
```

NB7 v2 tìm theo thứ tự:

1. `EVAL_DIR_OVERRIDE` nếu bạn set thủ công;
2. local experiment path trong current runtime;
3. các common path dưới `/content/drive/MyDrive`;
4. recursive search có giới hạn dưới `MyDrive/ML_Final`, `MyDrive/ML Final`, rồi `MyDrive`.

**Hai JSONL predictions là bắt buộc.** `evaluation_summary.json` chỉ dùng để đối chiếu metric và không còn là hard requirement.


In [ ]:
from src.data.runtime_paths import load_runtime_paths
from src.data.min2_experiment import scorer_ready_path

PATHS_CONFIG = ROOT / "configs/data_paths.min2_experiment.json"
paths = load_runtime_paths(repo_root=ROOT, config_path=PATHS_CONFIG)

# Optional override, ví dụ:
# EVAL_DIR_OVERRIDE = Path("/content/drive/MyDrive/ML_Final/min2_exp_v1/evaluation_min2_exp_v1")
EVAL_DIR_OVERRIDE = None

REQUIRED_PREDICTION_FILES = (
    "loo_predictions_valid.jsonl",
    "loo_predictions_test.jsonl",
)

def is_eval_dir(path: Path) -> bool:
    path = Path(path)
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_PREDICTION_FILES)

def mount_drive_if_colab():
    if not Path("/content").exists():
        return
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        return
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        print("Drive mount skipped:", type(exc).__name__, exc)

def bounded_find_eval_dirs(base: Path, max_depth: int = 6):
    """Find target folder without traversing arbitrarily deep Drive trees."""
    base = Path(base)
    if not base.is_dir():
        return []
    found = []
    base_depth = len(base.parts)
    for root, dirs, _files in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth >= max_depth:
            dirs[:] = []
        if root_path.name == "evaluation_min2_exp_v1" and is_eval_dir(root_path):
            found.append(root_path.resolve())
            dirs[:] = []
    return found

mount_drive_if_colab()

candidate_eval_dirs = []
if EVAL_DIR_OVERRIDE is not None:
    candidate_eval_dirs.append(Path(EVAL_DIR_OVERRIDE).expanduser().resolve())

candidate_eval_dirs.append((paths.scorer_ready_dir / "evaluation_min2_exp_v1").resolve())

common_drive_dirs = [
    Path("/content/drive/MyDrive/ML_Final/min2_exp_v1/evaluation_min2_exp_v1"),
    Path("/content/drive/MyDrive/ML_Final/evaluation_min2_exp_v1"),
    Path("/content/drive/MyDrive/ML Final/min2_exp_v1/evaluation_min2_exp_v1"),
    Path("/content/drive/MyDrive/ML Final/evaluation_min2_exp_v1"),
]
candidate_eval_dirs.extend(common_drive_dirs)

seen = set()
deduped = []
for p in candidate_eval_dirs:
    key = str(p)
    if key not in seen:
        seen.add(key)
        deduped.append(p)
candidate_eval_dirs = deduped

EVAL_DIR = next((p for p in candidate_eval_dirs if is_eval_dir(p)), None)

if EVAL_DIR is None:
    search_roots = [
        Path("/content/drive/MyDrive/ML_Final"),
        Path("/content/drive/MyDrive/ML Final"),
        Path("/content/drive/MyDrive"),
    ]
    for search_root in search_roots:
        matches = bounded_find_eval_dirs(search_root, max_depth=6)
        if matches:
            EVAL_DIR = matches[0]
            print("auto-discovered evaluation folder:", EVAL_DIR)
            break

print("scorer-ready:", paths.scorer_ready_dir)
print("eval dir    :", EVAL_DIR)

if EVAL_DIR is None:
    raise FileNotFoundError(
        "\nKhông tìm thấy loo_predictions_valid.jsonl + loo_predictions_test.jsonl.\n\n"
        "Nguyên nhân thường gặp: NB6B đã lưu evaluation vào /content/opisoverated/data/..., "
        "sau đó Colab runtime bị reset nên các file ephemeral đã mất.\n\n"
        "Cách khắc phục:\n"
        "1) Nếu runtime NB6B cũ vẫn còn: copy folder evaluation_min2_exp_v1 sang Google Drive.\n"
        "2) Nếu runtime cũ đã mất: cần chạy lại NB6B ít nhất tới LOO evaluation + Save artifacts, "
        "rồi copy folder đó sang Drive.\n"
        "3) Sau đó set EVAL_DIR_OVERRIDE nếu auto-discovery không tìm thấy.\n\n"
        "NB7 không thể tái dựng per-sample LOO deltas chỉ từ các metric summary."
    )

for name in ("evaluation_summary.json", *REQUIRED_PREDICTION_FILES):
    p = EVAL_DIR / name
    print(name, "exists=", p.is_file(), "size=", p.stat().st_size if p.is_file() else None)


### Nếu NB6B runtime cũ vẫn còn, persist artifacts ngay

Chạy cell sau **trong notebook/runtime NB6B cũ** một lần:

```python
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive", force_remount=False)

src = Path("/content/opisoverated/data/scorer_ready_min2_exp_v1/evaluation_min2_exp_v1")
dst = Path("/content/drive/MyDrive/ML_Final/min2_exp_v1/evaluation_min2_exp_v1")

assert src.is_dir(), src
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)

print("persisted:", dst)
```

Sau đó NB7 v2 sẽ tự tìm folder này. Nếu runtime NB6B đã bị reset và `src` không còn, phải chạy lại phần evaluation của NB6B vì per-sample `loo_deltas` không tồn tại trong metric summary.


## 2. Load predictions

Phần error analysis cốt lõi chỉ cần hai prediction JSONL. Join với scorer-ready records là **optional**: nếu scorer-ready dataset cũng mất sau runtime reset, NB7 vẫn chạy rank/margin/length analysis; chỉ các cột item-ID/category sẽ để trống.


In [ ]:
import pandas as pd
import numpy as np

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

summary_path = EVAL_DIR / "evaluation_summary.json"
summary = {}
if summary_path.is_file():
    with summary_path.open("r", encoding="utf-8") as f:
        summary = json.load(f)

predictions = {
    split: read_jsonl(EVAL_DIR / f"loo_predictions_{split}.jsonl")
    for split in ("valid", "test")
}

scorer_index = {}
for split in ("valid", "test"):
    scorer_file = scorer_ready_path(paths.scorer_ready_dir, split)
    if scorer_file.is_file():
        rows = read_jsonl(scorer_file)
        scorer_index[split] = {
            row["sample_id"]: row
            for row in rows
            if int(row.get("label", -1)) == 0
        }
        print(split, "scorer metadata join available:", len(scorer_index[split]))
    else:
        scorer_index[split] = {}
        print(split, "scorer metadata join unavailable:", scorer_file)

for split in ("valid", "test"):
    print(split, "LOO predictions =", len(predictions[split]))

if summary:
    print("\nNB6B metric summary found.")
    print(json.dumps({"loo_metrics": summary.get("loo_metrics")}, indent=2))
else:
    print("\nevaluation_summary.json not found; continuing from prediction JSONL only.")


## 3. Derive per-sample error-analysis fields

Ranking rule giữ đúng evaluator:

```python
sorted(indices, key=lambda i: (-delta[i], i))
```

Các field chính:

\[
rank_{GT} = \text{vị trí của swapped item trong ranking theo }\Delta
\]

\[
margin = \Delta_{Top1} - \Delta_{GT}
\]

- `gt_rank = 1`: localization đúng;
- `gt_rank = 2` + margin nhỏ: near-miss;
- `gt_rank = 2` + margin lớn: clear ranking error;
- `gt_rank >= 3`: hard localization failure.


In [ ]:
def enrich_prediction(row, scorer_record=None):
    out = dict(row)
    deltas = [float(x) for x in row["loo_deltas"]]
    n = len(deltas)
    gt = int(row["gt_swapped_item_index"])
    ranked = sorted(range(n), key=lambda i: (-deltas[i], i))
    gt_rank = ranked.index(gt) + 1
    pred = ranked[0]

    meta = (scorer_record or {}).get("negative_metadata") or {}
    items = list((scorer_record or {}).get("items") or [])

    out.update({
        "gt_rank": gt_rank,
        "top1_margin_over_gt": float(deltas[pred] - deltas[gt]),
        "top1_minus_top2_margin": (
            float(deltas[ranked[0]] - deltas[ranked[1]]) if n >= 2 else np.nan
        ),
        "gt_item_id": items[gt] if gt < len(items) else meta.get("replacement_item_id"),
        "predicted_item_id": items[pred] if pred < len(items) else None,
        "original_swapped_out_item_id": meta.get("original_item_id"),
        "replacement_item_id": meta.get("replacement_item_id"),
        "swap_category": meta.get("swap_category"),
        "all_item_ids": items,
        "ranked_indices": ranked,
        "ranked_deltas": [deltas[i] for i in ranked],
    })
    return out

enriched = {}
frames = {}

for split in ("valid", "test"):
    rows = []
    joined = 0
    for row in predictions[split]:
        record = scorer_index[split].get(row["sample_id"])
        joined += int(record is not None)
        rows.append(enrich_prediction(row, record))
    enriched[split] = rows
    frames[split] = pd.DataFrame(rows)
    print(split, "rows=", len(rows), "scorer metadata joined=", joined)

assert all(len(df) > 0 for df in frames.values())


## 4. Recompute core LOO metrics + random baseline


In [ ]:
def core_metrics(df):
    top1 = float((df["gt_rank"] == 1).mean())
    hit2 = float((df["gt_rank"] <= 2).mean())
    random_top1 = float(np.mean(1.0 / df["outfit_length"].astype(float)))
    random_hit2 = float(np.mean(np.minimum(2.0 / df["outfit_length"].astype(float), 1.0)))
    return {
        "n": int(len(df)),
        "top1": top1,
        "hit_at_2": hit2,
        "random_top1": random_top1,
        "random_hit_at_2": random_hit2,
        "top1_lift_over_random": top1 / random_top1,
        "hit2_lift_over_random": hit2 / random_hit2,
    }

for split in ("valid", "test"):
    print("\n", split.upper())
    print(json.dumps(core_metrics(frames[split]), indent=2))


## 5. Ground-truth rank distribution


In [ ]:
def rank_distribution(df):
    counts = df["gt_rank"].value_counts().sort_index()
    total = len(df)
    return pd.DataFrame([
        {"gt_rank": int(rank), "count": int(count), "fraction": float(count / total)}
        for rank, count in counts.items()
    ])

for split in ("valid", "test"):
    print("\n", split.upper())
    display(rank_distribution(frames[split]))


In [ ]:
import matplotlib.pyplot as plt

for split in ("valid", "test"):
    dist = rank_distribution(frames[split])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(dist["gt_rank"].astype(str), dist["fraction"])
    ax.set_xlabel("Ground-truth rank")
    ax.set_ylabel("Fraction of eligible negatives")
    ax.set_title(f"LOO GT-rank distribution — {split}")
    ax.set_ylim(0, 1)
    plt.show()


## 6. Rank-2 near-miss analysis

Đây là phần quan trọng nhất để giải thích gap Top-1 → Hit@2.

Ta đo:

\[
gap = \Delta_{Top1} - \Delta_{GT}
\]

và xem bao nhiêu case rank-2 có gap dưới `0.01 / 0.05 / 0.10 / 0.25`.


In [ ]:
MARGIN_THRESHOLDS = [0.01, 0.05, 0.10, 0.25]

def rank2_stats(df):
    r2 = df[df["gt_rank"] == 2].copy()
    if len(r2) == 0:
        return {"count": 0}
    gaps = r2["top1_margin_over_gt"].astype(float)
    result = {
        "count": int(len(r2)),
        "fraction_of_all": float(len(r2) / len(df)),
        "mean_gap": float(gaps.mean()),
        "median_gap": float(gaps.median()),
        "p90_gap": float(gaps.quantile(0.90)),
    }
    for threshold in MARGIN_THRESHOLDS:
        result[f"fraction_gap_le_{threshold:.2f}"] = float((gaps <= threshold).mean())
    return result

for split in ("valid", "test"):
    print("\n", split.upper())
    print(json.dumps(rank2_stats(frames[split]), indent=2))


## 7. Breakdown theo outfit length


In [ ]:
def length_breakdown(df):
    rows = []
    for length, group in df.groupby("outfit_length"):
        random_top1 = 1.0 / float(length)
        rows.append({
            "outfit_length": int(length),
            "n": int(len(group)),
            "top1": float((group["gt_rank"] == 1).mean()),
            "hit_at_2": float((group["gt_rank"] <= 2).mean()),
            "random_top1": random_top1,
            "top1_lift": float((group["gt_rank"] == 1).mean() / random_top1),
            "mean_gt_delta": float(group["gt_delta"].astype(float).mean()),
            "mean_rank2_gap": (
                float(group.loc[group["gt_rank"] == 2, "top1_margin_over_gt"].astype(float).mean())
                if (group["gt_rank"] == 2).any() else np.nan
            ),
            "hard_failure_fraction": float((group["gt_rank"] >= 3).mean()),
        })
    return pd.DataFrame(rows).sort_values("outfit_length")

for split in ("valid", "test"):
    print("\n", split.upper())
    display(length_breakdown(frames[split]))


## 8. Inspect representative errors


In [ ]:
DISPLAY_COLUMNS = [
    "sample_id",
    "outfit_length",
    "gt_rank",
    "gt_swapped_item_index",
    "predicted_problematic_index",
    "gt_delta",
    "predicted_top1_delta",
    "top1_margin_over_gt",
    "swap_category",
    "replacement_item_id",
    "predicted_item_id",
]

test_df = frames["test"].copy()

rank2 = test_df[test_df["gt_rank"] == 2].sort_values("top1_margin_over_gt")
hard = test_df[test_df["gt_rank"] >= 3].sort_values(
    ["gt_rank", "top1_margin_over_gt"], ascending=[False, False]
)

print("Rank-2 nearest misses:")
display(rank2[DISPLAY_COLUMNS].head(20))

print("\nRank-2 clearest errors:")
display(rank2[DISPLAY_COLUMNS].tail(20).sort_values("top1_margin_over_gt", ascending=False))

print("\nHard failures (GT rank >= 3):")
display(hard[DISPLAY_COLUMNS].head(30))


## 9. Optional category analysis

Chỉ chạy khi scorer-ready metadata vẫn tồn tại và NB7 join được `swap_category`.
Nếu dataset metadata đã mất sau Colab reset, phần này sẽ skip nhưng các analysis chính ở trên vẫn hợp lệ.


In [ ]:
for split in ("valid", "test"):
    df = frames[split]
    with_cat = df[df["swap_category"].notna()].copy()
    if with_cat.empty:
        print(split, ": no scorer metadata/category join; skip category breakdown.")
        continue

    cat = (
        with_cat.groupby("swap_category")
        .agg(
            n=("sample_id", "size"),
            top1=("gt_rank", lambda s: float((s == 1).mean())),
            hit_at_2=("gt_rank", lambda s: float((s <= 2).mean())),
            mean_gt_delta=("gt_delta", "mean"),
            mean_margin=("top1_margin_over_gt", "mean"),
        )
        .reset_index()
        .sort_values(["n", "top1"], ascending=[False, True])
    )
    print("\n", split.upper())
    display(cat.head(30))


## 10. Decision-oriented summary


In [ ]:
decision_rows = []
for split in ("valid", "test"):
    df = frames[split]
    r2 = df[df["gt_rank"] == 2]
    decision_rows.append({
        "split": split,
        "n": int(len(df)),
        "top1": float((df["gt_rank"] == 1).mean()),
        "hit_at_2": float((df["gt_rank"] <= 2).mean()),
        "rank2_fraction": float((df["gt_rank"] == 2).mean()),
        "rank3plus_fraction": float((df["gt_rank"] >= 3).mean()),
        "rank2_median_gap": float(r2["top1_margin_over_gt"].median()) if len(r2) else np.nan,
        "rank2_gap_le_0.05": float((r2["top1_margin_over_gt"] <= 0.05).mean()) if len(r2) else np.nan,
        "rank2_gap_le_0.10": float((r2["top1_margin_over_gt"] <= 0.10).mean()) if len(r2) else np.nan,
        "mean_gt_delta": float(df["gt_delta"].astype(float).mean()),
        "mean_top1_delta": float(df["predicted_top1_delta"].astype(float).mean()),
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)

print(
    "Interpretation:\n"
    "- Rank-2 lớn + gap nhỏ nhiều  -> LOO chủ yếu near-miss/ambiguous; ưu tiên confidence/top-2 UX.\n"
    "- Rank-2 lớn + gap lớn        -> scorer ranking không align tốt với swapped-item ground truth.\n"
    "- Rank-3+ đáng kể             -> localization failure sâu hơn; diagnosis-aware training đáng thử.\n"
    "- Degrade mạnh theo length    -> investigate pair-mean dilution / length-aware aggregation."
)


## 11. Save NB7 analysis artifacts


In [ ]:
OUTPUT_DIR = EVAL_DIR / "loo_error_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ("valid", "test"):
    frames[split].to_json(
        OUTPUT_DIR / f"loo_error_analysis_{split}.jsonl",
        orient="records",
        lines=True,
        force_ascii=False,
    )
    rank_distribution(frames[split]).to_csv(
        OUTPUT_DIR / f"gt_rank_distribution_{split}.csv", index=False
    )
    length_breakdown(frames[split]).to_csv(
        OUTPUT_DIR / f"length_breakdown_{split}.csv", index=False
    )

decision_df.to_csv(OUTPUT_DIR / "decision_summary.csv", index=False)

with (OUTPUT_DIR / "decision_summary.json").open("w", encoding="utf-8") as f:
    json.dump(decision_rows, f, ensure_ascii=False, indent=2)
    f.write("\n")

print("saved:", OUTPUT_DIR)
